# 이원 분산 분석

가정에서 재배하고 있는 네 가지 토마토 종자(A, B, C, D)에 대해 세 가지 종류의 비료 (11, 12, 13)를 사용하여 재배된 토마토 수를 조사하였다. 종자 및 비료 종류 간의 토마토 수의 차이가 있는지 유의수준 0.05하에서 검정하시오.
(단, 정규성, 등분산성에 만족한 데이터)
- 종자 (주 효과)
    - 귀무가설(H0): 종자 간의 토마토 수에 차이가 없다.
    - 대립가설(H1): 적어도 하나의 종자에서 토마토 수에 차이가 있다.
- 비료 (주 효과)
    - 귀무가설(H0): 비료 종류 간의 토마토 수에 차이가 없다.
    - 대립가설(H1): 적어도 하나의 비료 종류에서 토마토 수에 차이가 있다.
- 상호작용 효과:
    - 귀무가설(H0): 종자와 비료 간의 상호작용은 토마토 수에 영향을 미치지 않는다.
    - 대립가설(H1): 종자와 비료 간의 상호작용은 토마토 수에 영향을 미친다.


## 1. 기초

In [1]:
import pandas as pd
df = pd.read_csv("https://raw.githubusercontent.com/lovedlim/inf/main/p3/tomato.csv")
# df = pd.read_csv("tomato.csv")
print(df.head())
print(df.shape)

  종자  비료  토마토수
0  A  11    54
1  A  11    48
2  A  11    56
3  A  11    65
4  A  11    47
(120, 3)


### 이원 분산 분석

In [2]:
# anova 테이블
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('토마토수 ~ 종자 + 비료 + 종자:비료', data=df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
종자,3.0,4801.000000,1600.333333,18.297521,9.892196e-10
비료,1.0,877.812500,877.812500,10.036530,1.978360e-03
종자:비료,3.0,406.137500,135.379167,1.547867,2.061233e-01
Residual,112.0,9795.716667,87.461756,NaN,NaN


In [3]:
# 범주형 데이터 처리
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

model = ols('토마토수 ~ C(종자) + C(비료) + C(종자):C(비료)', data=df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
C(종자),3.0,4801.000000,1600.333333,18.757977,7.254117e-10
C(비료),2.0,1140.316667,570.158333,6.682993,1.835039e-03
C(종자):C(비료),6.0,725.350000,120.891667,1.417007,2.146636e-01
Residual,108.0,9214.000000,85.314815,NaN,NaN


In [5]:
# 일반표기법 format(지수표기법, '.10f')
print(format(7.254117e-10, '.10f'))
print(format(1.835039e-03, '.10f'))
print(format(2.146636e-01, '.10f'))

0.0000000007
0.0018350390
0.2146636000


In [6]:
# formula * 활용
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

# model = ols('토마토수 ~ C(종자) + C(비료) + C(종자):C(비료)', data=df).fit()
model = ols('토마토수 ~ C(종자) * C(비료)', data=df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
C(종자),3.0,4801.000000,1600.333333,18.757977,7.254117e-10
C(비료),2.0,1140.316667,570.158333,6.682993,1.835039e-03
C(종자):C(비료),6.0,725.350000,120.891667,1.417007,2.146636e-01
Residual,108.0,9214.000000,85.314815,NaN,NaN


## 2. 심화

### 사후검정

In [7]:
# 이원 분산 분석 수행
model = ols('토마토수 ~ C(종자) + C(비료) + C(종자):C(비료)', data=df).fit()
anova_lm(model)

,df,sum_sq,mean_sq,F,PR(>F)
C(종자),3.0,4801.000000,1600.333333,18.757977,7.254117e-10
C(비료),2.0,1140.316667,570.158333,6.682993,1.835039e-03
C(종자):C(비료),6.0,725.350000,120.891667,1.417007,2.146636e-01
Residual,108.0,9214.000000,85.314815,NaN,NaN


In [8]:
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Tukey HSD
tukey_summary1 = pairwise_tukeyhsd(df['토마토수'], df['종자'], alpha=0.05)
tukey_summary2 = pairwise_tukeyhsd(df['토마토수'], df['비료'].astype(str), alpha=0.05)
print(tukey_summary1)
print(tukey_summary2)

Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
     A      B   5.7333 0.1106 -0.8444  12.311  False
     A      C     12.1    0.0  5.5223 18.6777   True
     A      D     16.7    0.0 10.1223 23.2777   True
     B      C   6.3667 0.0616  -0.211 12.9444  False
     B      D  10.9667 0.0002   4.389 17.5444   True
     C      D      4.6 0.2679 -1.9777 11.1777  False
----------------------------------------------------
Multiple Comparison of Means - Tukey HSD, FWER=0.05 
group1 group2 meandiff p-adj   lower   upper  reject
----------------------------------------------------
    11     12    0.175 0.9973 -5.7831  6.1331  False
    11     13    6.625 0.0254  0.6669 12.5831   True
    12     13     6.45 0.0305  0.4919 12.4081   True
----------------------------------------------------


In [9]:
# Bonferroni
from scipy import stats
from statsmodels.stats.multicomp import MultiComparison

mc = MultiComparison(df['토마토수'], df['종자'])
bon_result = mc.allpairtest(stats.ttest_ind, method='bonf')
print(bon_result[0])

mc = MultiComparison(df['토마토수'], df['비료'].astype(str))
bon_result = mc.allpairtest(stats.ttest_ind, method='bonf')
print(bon_result[0])

Test Multiple Comparison ttest_ind 
FWER=0.05 method=bonf
alphacSidak=0.01, alphacBonf=0.008
group1 group2   stat   pval  pval_corr reject
---------------------------------------------
     A      B -2.3457 0.0224    0.1346  False
     A      C -4.9096    0.0       0.0   True
     A      D -7.0162    0.0       0.0   True
     B      C -2.3944 0.0199    0.1194  False
     B      D -4.2491 0.0001    0.0005   True
     C      D -1.7691 0.0821    0.4928  False
---------------------------------------------
Test Multiple Comparison ttest_ind 
FWER=0.05 method=bonf
alphacSidak=0.02, alphacBonf=0.017
group1 group2   stat   pval  pval_corr reject
---------------------------------------------
    11     12 -0.0691 0.9451       1.0  False
    11     13 -2.8722 0.0052    0.0157   True
    12     13  -2.411 0.0183    0.0548  False
---------------------------------------------


### [참고] 정규성, 등분산

In [10]:
from scipy.stats import shapiro

cond_tree_A = df['종자'] == 'A'
cond_tree_B = df['종자'] == 'B'
cond_tree_C = df['종자'] == 'C'
cond_tree_D = df['종자'] == 'D'

cond_fert_1 = df['비료'] == 11
cond_fert_2 = df['비료'] == 12
cond_fert_3 = df['비료'] == 13

print(shapiro(df[cond_tree_A & cond_fert_1]['토마토수']))
print(shapiro(df[cond_tree_A & cond_fert_2]['토마토수']))
print(shapiro(df[cond_tree_A & cond_fert_3]['토마토수']))

print(shapiro(df[cond_tree_B & cond_fert_1]['토마토수']))
print(shapiro(df[cond_tree_B & cond_fert_2]['토마토수']))
print(shapiro(df[cond_tree_B & cond_fert_3]['토마토수']))

print(shapiro(df[cond_tree_C & cond_fert_1]['토마토수']))
print(shapiro(df[cond_tree_C & cond_fert_2]['토마토수']))
print(shapiro(df[cond_tree_C & cond_fert_3]['토마토수']))

print(shapiro(df[cond_tree_D & cond_fert_1]['토마토수']))
print(shapiro(df[cond_tree_D & cond_fert_2]['토마토수']))
print(shapiro(df[cond_tree_D & cond_fert_3]['토마토수']))

ShapiroResult(statistic=np.float64(0.897816553535449), pvalue=np.float64(0.20729402301632316))
ShapiroResult(statistic=np.float64(0.9525045795563034), pvalue=np.float64(0.6981772706403775))
ShapiroResult(statistic=np.float64(0.9625476904883782), pvalue=np.float64(0.8144872716038309))
ShapiroResult(statistic=np.float64(0.9678788472402255), pvalue=np.float64(0.8705118004375336))
ShapiroResult(statistic=np.float64(0.9723742113682119), pvalue=np.float64(0.9119427430913412))
ShapiroResult(statistic=np.float64(0.9000301149374218), pvalue=np.float64(0.219263211589448))
ShapiroResult(statistic=np.float64(0.9434703107703121), pvalue=np.float64(0.5922336421612504))
ShapiroResult(statistic=np.float64(0.8953344131173363), pvalue=np.float64(0.1945690919149648))
ShapiroResult(statistic=np.float64(0.9231229200955566), pvalue=np.float64(0.38373908869822115))
ShapiroResult(statistic=np.float64(0.9585067948704918), pvalue=np.float64(0.7687207995864245))
ShapiroResult(statistic=np.float64(0.9392850171805

In [11]:
from scipy.stats import levene
print(levene(df[cond_tree_A]['토마토수'],
             df[cond_tree_B]['토마토수'],
             df[cond_tree_C]['토마토수'],
             df[cond_tree_D]['토마토수']))
print(levene(df[cond_fert_1]['토마토수'],
             df[cond_fert_2]['토마토수'],
             df[cond_fert_3]['토마토수']))

LeveneResult(statistic=np.float64(0.29736371082729113), pvalue=np.float64(0.827239106138081))
LeveneResult(statistic=np.float64(1.7801628913209462), pvalue=np.float64(0.17314691537302473))
